## 训练环境与项目日志初始化

In [5]:
import numpy as np
import json  # 导入json模块
import logging

from data import plot_data, format_dict, make_dirs, TrainingHistory
from controller import PIDController

train_model = True  # 是否训练模型
controller_type = "PID"  # 控制器类型：TD3 或 PPO
history: TrainingHistory = None

In [6]:
from datetime import datetime
import os
current_time = datetime.now().strftime("%Y%m%d_%H%M%S")
project_name = input("请输入加载/创建项目的名称 (父目录: .\\savedata) ").strip()# 创建保存模型的基础目录
project_path, ckpt_dir, plt_dir = make_dirs(project_name)
file_path = os.path.join(project_path, f'training_{current_time}.log')
project_path, ckpt_dir, plot_path = make_dirs(project_name)
if train_model:
    logging.basicConfig(filename=file_path, level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s') # 设置日志格式
logging.info("## 当前时间: %s", datetime.now())
logging.info("项目保存目录: %s", project_path)
logging.info("日志文件: %s", file_path)
logging.info("训练模式: %s", train_model)
print(f"日志文件: {file_path}")

日志文件: savedata\pidcs\training_20260404_110456.log


## 训练参数

In [35]:
from fx import zero, tolerance_smooth_reward

ENV_PARAMS = {
    'Ts': 0.001,  # 环境时间步长
    'T': 1.0,     # 每回合总时间
    'state0': np.array([0.0, 0.0, 0.0, 0.006, 0.0, 0.0]),  # 初始状态
    'obs_indices': [0, 2, 3, 5],  # 观测状态索引
    'x1_limit': 0.03,  # 状态 x1 的限制
    'use_dt_noise': False,  # 是否使用时间步长噪声
    'dt_noise_std': 0.1,  # 时间步长噪声标准差
    'delay_enabled': False,  # 是否启用动作延迟
    'delay_mean_steps': 0,  # 延迟均值步数
    'delay_std_steps': 0,   # 延迟标准差步数
    'include_dt_in_obs': False,  # 是否在观测中包含时间步长
    'include_delay_in_obs': False,  # 是否在观测中包含延迟
    'z_func': zero,  # 状态惩罚函数
    'r_func': tolerance_smooth_reward(1e-3),  # 奖励函数
    'f_func': zero,  # 外部激励函数
    # 'f_func': sin_wave(15000, 30*2*np.pi),  # 外部激励函数
}

PID_PARAMS = {
    'kp': 10000.0,  # 比例增益
    'ki': 0.0,    # 积分增益
    'kd':10.0,   # 微分增益
    'target': 0.0,  # 目标值
    'target_idx': 3,  # 目标状态索引
    'dt': ENV_PARAMS['Ts'],  # 时间步长
}

logging.info("环境参数:\n%s", json.dumps(format_dict(ENV_PARAMS), indent=4, ensure_ascii=False))
logging.info("PID 参数:\n%s", json.dumps(PID_PARAMS, indent=4, ensure_ascii=False))

## PID控制

In [ ]:
from env import build_env
env = build_env(ENV_PARAMS)
nc_recorder = env.run_episode(controller=None, state0=ENV_PARAMS['state0'], z_func=ENV_PARAMS['z_func'], f_func=ENV_PARAMS['f_func'])
nc_x_values=nc_recorder.as_numpy(keys='time_history').reshape(-1, 1)[:,[0,0,0,0,0,0]]
nc_y_values=nc_recorder.as_numpy(keys='state_history')[:,[0,1,2,3,4,5]]
c_recorder = env.run_episode(controller=PIDController(**PID_PARAMS), state0=ENV_PARAMS['state0'], z_func=ENV_PARAMS['z_func'], f_func=ENV_PARAMS['f_func'])
c_x_values=c_recorder.as_numpy(keys='time_history').reshape(-1, 1)
c_y_values=c_recorder.as_numpy(keys='state_history')[:,[0,1,2,3,4,5]]
plot_data(x_values=c_x_values, y_values=np.concatenate((c_y_values,nc_y_values[:,[3]]), axis=1), 
            legends=[('吸振器位移',),('无控制-主结构位移','PID控制-主结构位移'),('吸振器速度',),('主结构速度',),('吸振器加速度',),('主结构加速度',)], legend_loc='upper right',
            sub_shape=(3, 2), sub_group=[(0,), (6,3), (1,), (4,), (2,), (5,)],
            plot_title=f'{current_time}_初位移条件PID控制器响应', save_path=plot_path, show=True)
c_action_values = c_recorder.as_numpy(keys='action_history').reshape(-1, 1)
c_reward_values = c_recorder.as_numpy(keys='reward_history').reshape(-1, 1)
c_delay_time_values = c_recorder.as_numpy(keys='delay_time').reshape(-1, 1)
c_dt_values = c_recorder.as_numpy(keys='dt_history').reshape(-1, 1)
plot_data(x_values=c_x_values, y_values=np.concatenate((c_action_values, c_reward_values, c_delay_time_values, c_dt_values), axis=1),
            sub_shape=(2, 2), sub_group=[(0,), (1,), (2,), (3,)],
            legends=[('动作',), ('奖励',), ('延迟时间',), ('时间步长',)], legend_loc='upper right',
            plot_title=f'{current_time}_初位移条件PID控制器动作等', save_path=plot_path, show=True)
